## Apply Permissions

This notebook applies ACL entries to deployed Genie Spaces.

**What it does:**
1. Reads the deployment manifest
2. Loads exported permissions for each space
3. Applies permissions to target spaces
4. Reports any missing principals

**Prerequisites:**
- CAN_MANAGE on deployed Genie Spaces
- Principals (users/groups) must exist in target workspace

## Install Dependencies

In [ ]:
%pip install databricks-sdk>=0.76.0 --quiet
dbutils.library.restartPython()

## Setup Path Resolution

In [ ]:
import sys
import os

notebook_path = os.getcwd()
if notebook_path.startswith("/Workspace"):
    bundle_root = os.path.dirname(notebook_path)
else:
    bundle_root = os.path.dirname(os.path.abspath("__file__"))

helpers_path = os.path.join(bundle_root, "helpers")
if not os.path.exists(helpers_path):
    helpers_path = os.path.join(os.path.dirname(bundle_root), "helpers")

if helpers_path not in sys.path:
    sys.path.insert(0, os.path.dirname(helpers_path))

print(f"Bundle root: {bundle_root}")

## Read Parameters

In [ ]:
dbutils.widgets.text("volume_path", "", "Import Volume Path")
dbutils.widgets.text("apply_permissions", "true", "Apply Permissions (true/false)")

volume_path = dbutils.widgets.get("volume_path")
apply_permissions = dbutils.widgets.get("apply_permissions").lower() == "true"

if not volume_path:
    raise ValueError("volume_path parameter is required")

print(f"Volume path: {volume_path}")
print(f"Apply permissions: {apply_permissions}")

## Load Deployment Manifest

In [ ]:
import csv
import pandas as pd

manifest_path = os.path.join(volume_path, "deployment_manifest.csv")

if not os.path.exists(manifest_path):
    raise FileNotFoundError(
        f"Deployment manifest not found: {manifest_path}\n"
        "Run Tgt_02_Deploy_Genie_Spaces first."
    )

with open(manifest_path, "r", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    deployment_results = list(reader)

print(f"Loaded {len(deployment_results)} deployment results")
display(pd.DataFrame(deployment_results))

## Apply Permissions

In [ ]:
from databricks.sdk import WorkspaceClient
from helpers.permissions import apply_all_permissions

w = WorkspaceClient()
print(f"Connected to: {w.config.host}")

if not apply_permissions:
    print("\n⚠️  Permission application is disabled.")
    print("Set apply_permissions=true to enable.")
else:
    print("\nApplying permissions...")
    summary = apply_all_permissions(
        client=w,
        deployment_results=deployment_results,
        import_path=volume_path
    )

## Display Results

In [ ]:
if apply_permissions:
    print("\nPermission Summary:")
    print(f"  Total spaces processed:  {summary['total_spaces']}")
    print(f"  Spaces with permissions: {summary['spaces_with_permissions']}")
    print(f"  Permissions applied:     {summary['total_applied']}")
    print(f"  Permissions skipped:     {summary['total_skipped']}")
    print(f"  Errors:                  {summary['total_errors']}")
    
    if summary['details']:
        print("\nDetails by space:")
        for detail in summary['details']:
            status = "✓" if not detail.get('errors') else "⚠️"
            print(f"  {status} {detail['title']}: {detail.get('applied', 0)} applied")
            if detail.get('errors'):
                for err in detail['errors']:
                    print(f"      Error: {err}")

## Summary

In [ ]:
print("=" * 60)
print("PERMISSIONS COMPLETE")
print("=" * 60)

if apply_permissions:
    print(f"Permissions applied: {summary['total_applied']}")
    print(f"Errors:              {summary['total_errors']}")
else:
    print("Permission application was skipped.")

print("\nNext step: Run Tgt_04_Reconciliation to validate the migration.")